# FordA — training set

Exploration only. Nothing here feeds a result in the write-up; see [README.md](README.md).

Official UCR partition, training split. The test split stays untouched.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def repo_root() -> Path:
    """Walk up from the notebook's directory until src/config.yaml turns up."""
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "src" / "config.yaml").exists():
            return d
    raise FileNotFoundError("run this from inside the Assignment 4 tree")


ROOT = repo_root()
RAW = ROOT / "data" / "raw" / "FordA"
print(ROOT)

## Load

The UCR `.txt` files are whitespace-delimited: column 0 is the label, the remaining columns
are the series. Reshaped to the package convention `(n_series, n_channels, length)` used by
`src/representations/` — FordA is univariate, so the channel axis has length 1.

In [ ]:
def load_split(name: str = "FordA", split: str = "TRAIN"):
    """Return X of shape (n_series, n_channels, length) and integer-free labels y."""
    table = pd.read_csv(RAW / f"{name}_{split}.txt", sep=r"\s+", header=None)
    y = table.iloc[:, 0].to_numpy()
    X = table.iloc[:, 1:].to_numpy(dtype=np.float64)[:, None, :]
    return X, y


X_train, y_train = load_split(split="TRAIN")
X_train.shape, y_train.shape

## Shape

In [ ]:
n_series, n_channels, n_timestamps = X_train.shape

print(f"series      {n_series}")
print(f"channels    {n_channels}")
print(f"timestamps  {n_timestamps}")

classes, counts = np.unique(y_train, return_counts=True)
print()
print(f"classes     {len(classes)}")
for cls, count in zip(classes, counts):
    print(f"  {cls:+.0f}       {count:5d}  ({count / n_series:.1%})")

print()
print(f"missing     {int(np.isnan(X_train).sum())}")

## One series per class

Two dense 500-point signals overlaid would be unreadable, so this is small multiples on a
shared y-axis — the panels are directly comparable, and each panel's title names its class,
which is why there is no legend. The example index is drawn with a fixed seed so re-running
shows the same series.

In [ ]:
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
GRID = "#e5e4e0"
CLASS_COLOR = {-1.0: "#2a78d6", 1.0: "#eb6834"}  # validated categorical slots 1 and 2

rng = np.random.default_rng(42)

fig, axes = plt.subplots(
    len(classes), 1, figsize=(11, 2.5 * len(classes)), sharex=True, sharey=True
)
fig.patch.set_facecolor(SURFACE)

for ax, cls in zip(np.atleast_1d(axes), classes):
    idx = int(rng.choice(np.flatnonzero(y_train == cls)))
    ax.plot(X_train[idx, 0, :], lw=1.0, color=CLASS_COLOR[cls], solid_capstyle="round")

    ax.set_facecolor(SURFACE)
    ax.set_title(
        f"class {cls:+.0f}    series #{idx}", loc="left", fontsize=11, color=INK, pad=8
    )
    ax.set_ylabel("engine noise (z)", color=INK_MUTED, fontsize=9)
    ax.grid(True, axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)

np.atleast_1d(axes)[-1].set_xlabel("timestep", color=INK_MUTED, fontsize=9)
fig.suptitle(
    "FordA train — one example series per class", x=0.125, ha="left", fontsize=13, color=INK
)
fig.tight_layout()
plt.show()